# Task 4 — Gold Layer: Dimensional Modeling

Builds the Gold star schema from Silver as the sole source of truth: `fact_bookings` (one row per flight-segment booking line), four dimension tables (`dim_passenger`, `dim_route`, `dim_airline`, `dim_geography`), the `vw_annual_booking_kpis` view, and a Gold-vs-Silver reconciliation report with dimension key uniqueness validation. Passenger geography and airline geography are kept distinct throughout.

In [0]:
# -----------------------------------------------------------------------------
# Gold config + conformed dimensions (Silver is the sole source of truth)
# -----------------------------------------------------------------------------
from pyspark.sql import functions as F

try:
    catalog = dbutils.widgets.get("catalog_name")
except Exception:
    dbutils.widgets.text("catalog_name", "dev")
    catalog = dbutils.widgets.get("catalog_name")

SILVER = f"{catalog}.silver"
GOLD = f"{catalog}.gold"
OPS = f"{catalog}.ops"

# --- dim_geography: one row per nation, including its parent region ---
nation = spark.table(f"{SILVER}.nation")
region = spark.table(f"{SILVER}.region")
dim_geography = (nation.join(region, F.col("n_regionkey") == F.col("r_regionkey"), "left")
    .select(F.col("n_nationkey").alias("geography_key"),
            F.col("n_nationkey").alias("nation_id"),
            F.col("n_name").alias("nation_name"),
            F.col("n_regionkey").alias("region_id"),
            F.col("r_name").alias("region_name")))
dim_geography.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_geography")
print(f"dim_geography:    {dim_geography.count()} rows (one per nation)")

# --- dim_passenger: one row per passenger, with PASSENGER geography denormalized ---
passengers = spark.table(f"{SILVER}.passengers")
dim_passenger = (passengers.join(dim_geography, F.col("pa_nationkey") == F.col("geography_key"), "left")
    .select(F.col("pa_passengerid").alias("passenger_key"),
            F.col("pa_passengerid").alias("passenger_id"),
            F.col("pa_name").alias("passenger_name"),
            F.col("pa_address").alias("address"),
            F.col("pa_phone").alias("phone"),
            F.col("pa_loyaltypoints").alias("loyalty_points"),
            F.col("pa_travelpurpose").alias("travel_purpose"),
            F.col("pa_comment").alias("passenger_comment"),
            F.col("geography_key").alias("home_geography_key"),
            F.col("nation_name").alias("home_nation"),
            F.col("region_name").alias("home_region")))
dim_passenger.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_passenger")
print(f"dim_passenger:    {dim_passenger.count()} rows (one per passenger)")

# --- dim_airline: one row per airline, with AIRLINE geography (kept distinct from passenger geography) ---
airline = spark.table(f"{SILVER}.airline")
dim_airline = (airline.join(dim_geography, F.col("al_nationkey") == F.col("geography_key"), "left")
    .select(F.col("al_airlineid").alias("airline_key"),
            F.col("al_airlineid").alias("airline_id"),
            F.col("al_name").alias("airline_name"),
            F.col("al_address").alias("airline_address"),
            F.col("al_phone").alias("airline_phone"),
            F.col("al_rating").alias("airline_rating"),
            F.col("al_comment").alias("airline_comment"),
            F.col("geography_key").alias("airline_geography_key"),
            F.col("nation_name").alias("airline_nation"),
            F.col("region_name").alias("airline_region")))
dim_airline.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_airline")
print(f"dim_airline:      {dim_airline.count()} rows (one per airline)")

# --- dim_route: one row per route ---
route = spark.table(f"{SILVER}.route")
dim_route = (route.select(F.col("ro_routeid").alias("route_key"),
            F.col("ro_routeid").alias("route_id"),
            F.col("ro_name").alias("route_name"),
            F.col("ro_aircraft").alias("aircraft"),
            F.col("ro_airline_brand").alias("airline_brand"),
            F.col("ro_routetype").alias("route_type"),
            F.col("ro_capacity").alias("capacity"),
            F.col("ro_cabinconfig").alias("cabin_config"),
            F.col("ro_baseprice").alias("base_price"),
            F.col("ro_comment").alias("route_comment")))
dim_route.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_route")
print(f"dim_route:        {dim_route.count()} rows (one per route)")

In [0]:
# -----------------------------------------------------------------------------
# fact_bookings: one row per flight-segment booking line (grain = booking line)
# -----------------------------------------------------------------------------
fb = spark.table(f"{SILVER}.flight_booking")
bk = spark.table(f"{SILVER}.booking")
dp = spark.table(f"{GOLD}.dim_passenger")
dr = spark.table(f"{GOLD}.dim_route")
da = spark.table(f"{GOLD}.dim_airline")

# Pre-rename join partners to avoid column ambiguity (fb_ prefixed source kept explicit)
bk_r = bk.select(F.col("bk_bookingid").alias("booking_id"),
                 F.col("bk_passengerid").alias("passenger_key"),
                 F.col("bk_bookingdate").alias("booking_date"),
                 F.col("bk_flightdate").alias("flight_date"),
                 F.col("bk_status").alias("booking_status"),
                 F.col("bk_bookingpriority").alias("booking_priority"),
                 F.col("bk_agent").alias("booking_agent"))
dp_r = dp.select(F.col("passenger_key").alias("_dp_key"),
                 F.col("passenger_name"), F.col("travel_purpose"),
                 F.col("home_geography_key"), F.col("home_nation"), F.col("home_region"))
dr_r = dr.select(F.col("route_key").alias("_dr_key"),
                 F.col("route_name"), F.col("aircraft"), F.col("route_type"),
                 F.col("base_price"))
da_r = da.select(F.col("airline_key").alias("_da_key"),
                 F.col("airline_name"), F.col("airline_geography_key"),
                 F.col("airline_nation"), F.col("airline_region"))

fact_bookings = (fb.join(bk_r, F.col("fb_bookingid") == F.col("booking_id"), "left")
                 .join(dp_r, F.col("passenger_key") == F.col("_dp_key"), "left")
                 .join(dr_r, F.col("fb_routeid") == F.col("_dr_key"), "left")
                 .join(da_r, F.col("fb_airlineid") == F.col("_da_key"), "left")
                 .select(
                     # -- natural key of the grain: one row per booking line --
                     F.col("fb_bookingid").alias("booking_id"),
                     F.col("fb_linenumber").alias("line_number"),
                     # -- dimension keys (star-schema FKs) --
                     F.col("passenger_key"),
                     F.col("fb_routeid").alias("route_key"),
                     F.col("fb_airlineid").alias("airline_key"),
                     # -- measures --
                     F.col("fb_seats").alias("seats"),
                     F.col("fb_ticketprice").alias("ticket_price"),
                     F.col("fb_discount").alias("discount_rate"),
                     F.col("fb_tax").alias("tax_rate"),
                     # Metric Definitions (official): Net revenue (before tax) = fb_ticketprice * (1 - fb_discount)
                     (F.col("fb_ticketprice") * (1 - F.col("fb_discount"))).alias("net_revenue"),
                     # -- dates --
                     F.col("booking_date"), F.col("flight_date"),
                     F.col("fb_departuredate").alias("departure_date"),
                     F.col("fb_commitdate").alias("commit_date"),
                     F.col("fb_arrivaldate").alias("arrival_date"),
                     # -- booking attributes --
                     F.col("booking_status"), F.col("booking_priority"),
                     F.col("booking_agent"),
                     # -- segment attributes --
                     F.col("fb_refundflag").alias("refund_flag"),
                     F.col("fb_segmentstatus").alias("segment_status"),
                     F.col("fb_cabinclass").alias("cabin_class"),
                     # -- route / passenger / airline attributes --
                     F.col("route_name"), F.col("route_type"), F.col("aircraft"),
                     F.col("passenger_name"), F.col("travel_purpose"),
                     # -- PASSENGER geography (who travels) --
                     F.col("home_geography_key"), F.col("home_nation"), F.col("home_region"),
                     # -- AIRLINE geography (who operates) --
                     F.col("airline_geography_key"), F.col("airline_nation"), F.col("airline_region")))

fact_bookings.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.fact_bookings")
print(f"fact_bookings:    {fact_bookings.count()} rows (one per flight-segment booking line)")
display(fact_bookings.limit(5))

In [0]:
# -----------------------------------------------------------------------------
# vw_annual_booking_kpis: annual net revenue, distinct bookings, active passengers,
# and average booking value (KPI view over the fact table)
# -----------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_annual_booking_kpis AS
SELECT year(booking_date)                                        AS booking_year,
       ROUND(SUM(net_revenue), 2)                                AS annual_net_revenue,
       COUNT(DISTINCT booking_id)                                AS distinct_bookings,
       COUNT(DISTINCT passenger_key)                             AS active_passengers,
       ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2)   AS avg_booking_value
FROM {GOLD}.fact_bookings
GROUP BY year(booking_date)
""")
print(f"Created view {GOLD}.vw_annual_booking_kpis")
display(spark.sql(f"SELECT * FROM {GOLD}.vw_annual_booking_kpis ORDER BY booking_year"))

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- Gold vs Silver reconciliation + dimension key uniqueness validation
-- SQL approach: one SELECT per check, combined with UNION ALL per table
-- -----------------------------------------------------------------------------

CREATE OR REPLACE TABLE IDENTIFIER(:catalog_name || '.ops.gold_reconciliation') AS
WITH checks AS (
    -- 1) Row-count reconciliation: fact rows must equal Silver flight_booking rows
    SELECT 'fact_bookings row_count' AS check_name,
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings'))    AS gold_value,
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.silver.flight_booking')) AS silver_value

    UNION ALL

    -- 2) Revenue reconciliation: Gold net revenue vs an INDEPENDENT recomputation from Silver
    --    (formula applied directly on Silver columns, not read back from Gold)
    --    Metric Definitions (official): Net revenue (before tax) = SUM(fb_ticketprice * (1 - fb_discount))
    SELECT 'fact_bookings net_revenue_total',
           (SELECT CAST(ROUND(SUM(net_revenue), 2) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')),
           (SELECT CAST(ROUND(SUM(fb_ticketprice * (1 - fb_discount)), 2) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.silver.flight_booking'))

    UNION ALL

    -- 3) Dimension key uniqueness: distinct keys must equal total rows in every dimension
    SELECT 'dim_geography key_uniqueness',
           (SELECT CAST(COUNT(DISTINCT geography_key) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_geography')),
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_geography'))

    UNION ALL

    SELECT 'dim_passenger key_uniqueness',
           (SELECT CAST(COUNT(DISTINCT passenger_key) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_passenger')),
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_passenger'))

    UNION ALL

    SELECT 'dim_airline key_uniqueness',
           (SELECT CAST(COUNT(DISTINCT airline_key) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_airline')),
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_airline'))

    UNION ALL

    SELECT 'dim_route key_uniqueness',
           (SELECT CAST(COUNT(DISTINCT route_key) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_route')),
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.dim_route'))

    UNION ALL

    -- 4) Fact composite key uniqueness: (booking_id, line_number) must be unique
    SELECT 'fact_bookings composite_key_uniqueness',
           (SELECT CAST(COUNT(DISTINCT booking_id, line_number) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')),
           (SELECT CAST(COUNT(*) AS DOUBLE) FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings'))
)
SELECT check_name,
       gold_value,
       silver_value,
       ROUND(gold_value - silver_value, 2) AS difference,
       CASE WHEN ABS(gold_value - silver_value) <= 0.01 THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks;

SELECT * FROM IDENTIFIER(:catalog_name || '.ops.gold_reconciliation') ORDER BY check_name;

-- Fail the cell if any check failed (equivalent of the old raise Exception)
SELECT CASE
         WHEN COUNT(CASE WHEN status = 'FAIL' THEN 1 END) = 0
           THEN 'Gold reconciliation PASSED: Gold agrees with Silver (row count, revenue, and all dimension keys unique).'
         ELSE raise_error('GOLD RECONCILIATION FAILED for: ' ||
              concat_ws(', ', collect_list(CASE WHEN status = 'FAIL' THEN check_name END)))
       END AS result
FROM IDENTIFIER(:catalog_name || '.ops.gold_reconciliation');

In [0]:
%sql
select * from dev.ops.gold_reconciliation